# Ingestion & audit des données issues de Tiktok

##### Objectif
parser le JSON brut et inventorier chaque section: 
- posts, 
- commentaires, 
- historique de visionnage, 
- recherches, 
- hashtags, 
- connexions. 
Documenter ce qui est utile, ce qui est vide, et ce qui est inutilisable

Le 16/05/2026
-> Gora DIEYE

In [29]:
import json
import pandas as pd

In [38]:
import os
os.getcwd()

'C:\\Users\\miste\\Desktop'

In [41]:
os.chdir('C:\\Users\\miste\\Desktop\\Data_Eng_Project\\Amul_Xalaat_Growth_BI_Analytics')

In [42]:
os.getcwd()

'C:\\Users\\miste\\Desktop\\Data_Eng_Project\\Amul_Xalaat_Growth_BI_Analytics'

In [43]:
# Ouverture du fichier en mode lecture ('r')
with open('data/raw/tiktok/user_data_tiktok.json', 'r', encoding='utf-8') as data:
    tiktok_data = json.load(data)

## Audit de la structure

In [8]:
# Récupération des clés de premier niveau
cles = list(tiktok_data.keys())

print(cles)

['Comment', 'Likes and Favorites', 'Post', 'Your Activity']


### Clé de premier niveau
les clés de premier niveau nous montre les éléments principaux de notre dataset json
- `comment` -> les commentaires 
- `Likes and Favorites` -> les mentions j'aime et mes favoris
- `Post` -> les publications
- `Your Activity` -> résumé de mes activités sur Tiktok

In [9]:
subkey_comment = list(tiktok_data["Comment"].keys())
print(subkey_comment)

['Comments']


In [10]:
sub_subkey_comments = list(tiktok_data["Comment"]['Comments'].keys())
print(sub_subkey_comments)

['App', 'CommentsList']


In [45]:
commentsList = tiktok_data["Comment"]['Comments']['CommentsList']
print(commentsList[:5])
print(len(commentsList))

[{'date': '2026-05-11 18:55:39', 'comment': 'Jaajef! Content que ça aide', 'photo': 'N/A', 'video': 'N/A', 'url': ''}, {'date': '2026-05-10 21:59:14', 'comment': 'Ah oui c’est hilarant car tu te rends compte que ce sont des vérités simples mais qui nous échappent souvent😁', 'photo': 'N/A', 'video': 'N/A', 'url': ''}, {'date': '2026-05-10 15:09:36', 'comment': '🙌🏾❤️', 'photo': 'N/A', 'video': 'N/A', 'url': ''}, {'date': '2026-05-02 09:53:43', 'comment': 'Detwaay! Ligeeyal nak😁', 'photo': 'N/A', 'video': 'N/A', 'url': ''}, {'date': '2026-05-02 09:51:59', 'comment': 'Respect Seydina🤝', 'photo': 'N/A', 'video': 'N/A', 'url': ''}]
1422


In [12]:
print(commentsList[6])

{'date': '2026-05-01 23:26:00', 'comment': 'La base 🤝', 'photo': 'N/A', 'video': 'N/A', 'url': ''}


In [13]:
commentsAPP = tiktok_data["Comment"]['Comments']['App']
print((commentsAPP))


1


### structure des commentaires
Le premier clé du dictionnaire de base est `Comment`
Ce dernier a comme clé `Comments` qui, à son tour a comme clé `App` & `CommentsList`
CommentList contient tous les commentaires avec 5 champs :

`date` → horodatage du commentaire

`comment` → texte du commentaire

`photo et video` → semblent vides (N/A)

`url` → vide également

- Les champs photo, vidéo et url ne servent pas ici car ils sont vide.
- APP est int à la valeur 1, dont on se sait pas son utilité pour le moment

In [14]:
subkey_likes_favorites = list(tiktok_data["Likes and Favorites"].keys())
print(subkey_likes_favorites)

['Collection', 'Favorite Collection', 'Favorite Comment', 'Favorite Effects', 'Favorite Hashtags', 'Favorite Location', 'Favorite Sounds', 'Favorite Videos', 'Like List']


In [15]:
for element in subkey_likes_favorites:
    list_of_keys = list(tiktok_data["Likes and Favorites"][element].keys())
    if len(list_of_keys) > 0:
        for fav_key in list_of_keys:
            sub_fav_likes_favorites = tiktok_data["Likes and Favorites"][element][fav_key]
            if sub_fav_likes_favorites and isinstance(sub_fav_likes_favorites, list):
                print(f"{fav_key}  {len (sub_fav_likes_favorites)} éléments -> {sub_fav_likes_favorites[0]}")
            

FavoriteSoundList  78 éléments -> {'Date': '2026-04-08 19:58:59', 'Link': 'https://m.tiktok.com/h5/share/music/7583508856421042966.html'}
FavoriteVideoList  394 éléments -> {'Date': '2026-05-10 14:31:03', 'Link': 'https://www.tiktokv.com/share/video/7637512279302458645/'}
ItemFavoriteList  1974 éléments -> {'date': '2026-05-11 18:57:47', 'link': 'https://www.tiktokv.com/share/video/7630976266094578965/'}


# Likes and Favorites 
2em clé de notre dictionnaire
La majorité des champs sont vides. Sauf pour 3

- `FavoriteSoundList`: 78 éléments

- `FavoriteVideoList `: 394 éléments

- `ItemFavoriteList  `: 1974 éléments

FavoriteSoundList et FavoriteVideoList → ce sont mes comportements de consommation, pas de création. Peu utiles directement.
ItemFavoriteList → les vidéos likées. Même constat.

### Les publications: `Post`

3em clé de notre dictionnaire de données json

In [16]:
subkey_posts = list(tiktok_data["Post"].keys())
print(subkey_posts)

['Posts', 'Recently Deleted Posts', 'Story']


In [17]:
print(list(tiktok_data["Post"]['Posts'].keys()))
print(list(tiktok_data["Post"]['Recently Deleted Posts'].keys()))
print(list(tiktok_data["Post"]['Story'].keys()))

['VideoList']
['PostList']
[]


In [18]:
videoList = tiktok_data["Post"]['Posts']['VideoList']
delPostList = tiktok_data["Post"]['Recently Deleted Posts']['PostList']
print(f"{len(delPostList)} supprimées: liste: {delPostList}")

1 supprimées: liste: [{'Date': '2026-05-02 20:55:46', 'DateDeleted': '2026-05-02 20:55:53', 'Link': 'https://www.tiktok.com/login/?redirect_url=http%3A%2F%2Fwebapp-useastred.eu.tiktok.com%2F06d23414d9385c8f217379480a558e2d%2F6a103d3a%2Ftos-no1a-ve-0068-no%2FoABPZik2AOkIAByJEgZCwM5jXliCf08ywQA2IS%3Fa%3D1988%26bti%3DdXB2cmJ2d3dodHVwcndvYndndXYxcm53ZmJya2BoXlxwcmIrb2NebV5eXGJyaHFmOg%253D%253D%26%26bt%3D361%26ft%3Dcv1cNpz7ThWHKRIPLGZmo0P%26mime_type%3Dvideo_mp4%26rc%3DNXZpTGRTaFBnKWh4cWZuZGJ2eGgxdWlkc2RkYmh4bndsQCllNzo8Zzg3O2Q3OWgzZWU7ZylqO245OGo6PHVnMzNqNzd5eVNsa3ZpcUQ6am9qcGxccHFxYm5vamxxaVxxYW9wK2xocWA0Ll9eYGFjLi8vYWEvYS42OmMtNmIxZjNsZDM0LS1pMS0vOg%253D%253D%26l%3D2026051211250282C5ACE1BDA8055D7A48%26ply_type%3D3%26policy%3DeyJ2bSI6MywidWlkIjoiNzQ1NTk4MTcwMzM1NTQ5OTU0MiJ9%26btag%3D400088000', 'Likes': '0', 'ContentDisclosure': '', 'AIGeneratedContent': '', 'Sound': 'original sound - amul_xalaat', 'Location': 'N/A', 'Title': 'N/A', 'AddYoursText': 'N/A', 'AlternateText': 'N/A', 'CoverIma

Concernant les publications, la clé de base `Post` a comme sous-clés:

- `Posts`: les publications , 

- `Recently Deleted Posts`: les publications récemment supprimées, dont une seule a été listée

- `Story`: les stories qui sont vides

Ce qui nous sera utile ici est `videoList` contenant la liste de toutes les vidéos que j'ai publiées

#### Exploration de VideoList 

In [19]:
print(len(videoList))

156


In [20]:
print(videoList[0])

{'Date': '2026-05-01 12:08:51', 'Link': 'https://video-no1a.tiktokv.eu/storage/v1/tos-no1a-v-0068-no/o4AAniEEEFgp459ftlFk0ZDebvbDnEACDBJRA3?a=1233&bti=dXB2cmJ2d3dodHVwcndvYndndXYxcm53ZmJya2BoXlxwcmIrb2NebV5eXGJyaHFmOg%3D%3D&&bt=4629&ft=cpOdtEz7ThWHKRIPLGZmo0P&mime_type=video_mp4&rc=ajo6b3U5cng8OjMzbzczNUBpajo6b3U5cng8OjMzbzczNUBhcWhkMmQ0YW5hLS1kMTFzYSNhcWhkMmQ0YW5hLS1kMTFzcw%3D%3D&l=2026051211250282C5ACE1BDA8055D7A48&x-tos-algorithm=v2&x-tos-authkey=5bf25627da095a5cba28ace592de46cc&x-tos-expires=1779449142&x-tos-signature=cFePl8fY_tgNBm2l16rF726fcAc&btag=e00048000', 'Likes': '1814', 'WhoCanView': 'Everyone', 'AllowComments': 'Yes', 'AllowStitches': 'Yes', 'AllowDuets': 'No', 'AllowStickers': 'Yes', 'AllowSharingToStory': 'Yes', 'ContentDisclosure': '', 'AIGeneratedContent': '', 'Sound': 'original sound - amul_xalaat', 'Location': '', 'Title': 'N/A', 'AddYoursText': 'N/A', 'AlternateText': 'N/A', 'CoverImage': 'https://p16-common-sign.tiktokcdn-eu.com/tos-no1a-p-0037-no/okC9VFD4QFAbQuB9

In [21]:
print(len(tiktok_data["Post"]['Posts']['VideoList']))

156


variables utiles à notre étude:

`Date`: La date et l'heure de publicatoin. Example: 2026-05-01 12:08:51

`Likes`: le nombre de like de la publication: Example: 1814 pour la dernière publication

`Sound`: Le son de la vidéo: Example : original sound - amul_xalaat

`Location`: Lieu: pourrait être utile utérieurement mais est vide donc on l'élimine

`Title`: N/A Vide aussi

`WhoCanView` — pertinent pour comprendre la portée

`AllowComments` — indicateur de paramétrage d'engagement

`Link`, `CoverImage`, `AddYoursText`, `AlternateText`, `NumberOfCollections` — ce sont des métadonnées techniques sans valeur analytique.

### Exploration de la dernière clé de notre dictionnaire de base: Your Activity

In [22]:
subkey_your_activity = list(tiktok_data["Your Activity"].keys())
print(subkey_your_activity)

['Activity Summary', 'Ad Interests', 'Ads Visit History', 'Donation', 'Fundraiser', 'Hashtag', 'Instant Form Ads Responses', 'Login History', 'Mini Drama Watch History', 'Off TikTok Activity', 'Purchases', 'Reposts', 'Searches', 'Share History', 'Status', 'Stickers', 'Watch History']


In [28]:
print(tiktok_data["Your Activity"]['Activity Summary']['ActivitySummaryMap'])

{'note': 'data may have delays of up to several days', 'videosCommentedOnSinceAccountRegistration': 1424, 'videosSharedSinceAccountRegistration': 42, 'videosWatchedToTheEndSinceAccountRegistration': 6600}


In [44]:
print(tiktok_data["Your Activity"]['Off TikTok Activity'])

{'OffTikTokActivityDataList': None}


In [24]:
print(type(tiktok_data["Your Activity"]['Hashtag']))
print(tiktok_data["Your Activity"]['Hashtag']['HashtagList'][0])
print(len(tiktok_data["Your Activity"]['Hashtag']['HashtagList']))

<class 'dict'>
{'HashtagName': 'justice', 'HashtagLink': 'https://www.tiktok.com/share/challenge/7455981703355499542/'}
135


In [79]:
for cle, valeur in tiktok_data["Your Activity"]['Hashtag'].items():
    print(f"{cle} : {valeur[1]}")


HashtagList : {'HashtagName': 'life', 'HashtagLink': 'https://www.tiktok.com/share/challenge/7455981703355499542/'}


In [119]:
print(tiktok_data["Your Activity"]['Login History']["LoginHistoryList"][0])
print(len(tiktok_data["Your Activity"]['Login History']["LoginHistoryList"]))

{'Date': '2025-11-13 13:28:48', 'IP': '41.83.162.178', 'DeviceModel': 'iPhone16,2', 'DeviceSystem': 'iOS 18.5', 'NetworkType': 'Wi-Fi', 'Carrier': ''}
2674


In [123]:
print(len(tiktok_data["Your Activity"]['Watch History']["VideoList"]))
print(tiktok_data["Your Activity"]['Watch History']["VideoList"][0])

22947
{'Date': '2026-05-07 01:54:27', 'Link': 'https://www.tiktokv.com/share/video/7626100665563041046/'}


In [99]:
print(tiktok_data["Your Activity"]['Status']['Status List'][1])
print(len(tiktok_data["Your Activity"]['Status']['Status List']))

{'Resolution': '1290*2796', 'App Version': '40.1.0', 'IDFA': '', 'GAID': '', 'Android ID': '', 'IDFV': '4C40DAB1-655C-4CF3-B804-282FB76A4292', 'UID': 7455981703355499542, 'DID': '7428339531251942944', 'Web ID': ''}
176


In [120]:
print(tiktok_data["Your Activity"]['Reposts']['RepostList'][1])
print(len(tiktok_data["Your Activity"]['Reposts']['RepostList']))


{'Date': '2026-04-14 01:38:20', 'Link': 'https://www.tiktokv.com/share/video/7616739186611146017/'}
14


In [121]:
print(tiktok_data["Your Activity"]['Searches']['SearchList'][0])
print(len(tiktok_data["Your Activity"]['Searches']['SearchList']))

{'Date': '2025-02-17 16:11:11', 'SearchTerm': 'creating search insights'}
91


In [111]:
print(len(tiktok_data["Your Activity"]['Login History']["LoginHistoryList"]))

2674


### Le dictionnaire "Your Activity" présente mes activités sur Tiktok.
Les éléments qui seraient utile pour notre étude seraient:

`Activity Summary`: Le résumé de mes activités sur tiktok 

`Hashtag`: les hashtags que j'utilise 

`Reposts`: les posts que je repost 

`Searches`: ce que je recherche: infos pas trés utile 

`Watch History`: historique de mes vidéos regardées (avec ça, on peut voir le rapport entre ma consommation de vidéo et l'affluence sur mes publications

`WhoCanView` — pertinent pour comprendre la portée

`AllowComments` — indicateur de paramétrage d'engagement

Et on note aussi les champs clairement inutilisables pour notre analyse : Link, CoverImage, AddYoursText, AlternateText, NumberOfCollections — ce sont des métadonnées techniques sans valeur analytique.

| Section             | Enregistrements | Champs utiles | Statut |
|---------------------|----------------|---------------|--------|
| commentsList        | 1422           | date, comment | ✅     |
| videoList           | 156            | date, Likes, Sound, WhoCanView, AllowComments| ✅     |
| FavoriteSoundList   | 78            | date, Link|   ⚠️    |
| FavoriteVideoList   | 394            | date, Link| ⚠️    |
| ItemFavoriteList    | 1974            | date, Link| ⚠️     |
| Hashtag             | 135            | HashtagName, HashtagLink| ⚠️     |
| Watch History       | 22947        | Date, Link | ✅     |
| Login History       | 2674        | Date, IP, NetworkType | ✅     |
| Status     | 176        | Activity Summary, Reposts, Searches, Watch History, WhoCanView, AllowComments | ❌     |
| Reposts     | 14        | Date, Link | ✅     |
| Searches     | 91        |Date, SearchTerm | ✅     |